# Plot NVDA from Prepared Dataset

Load split-adjusted NVDA data from our momentum dataset and chart it.

In [1]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
# Load the prepared momentum dataset
print("Loading momentum dataset...")
df_all = pd.read_parquet('data/momentum_prepared/momentum_data.parquet')
print(f"Loaded {len(df_all):,} rows")
print(f"Date range: {df_all['date'].min()} to {df_all['date'].max()}")

Loading momentum dataset...
Loaded 13,495,461 rows
Date range: 2020-12-09 00:00:00 to 2025-12-04 00:00:00


In [3]:
# Filter for NVDA only
df_nvda = df_all[df_all['ticker'] == 'NVDA'].copy()
df_nvda = df_nvda.sort_values('date')

print(f"NVDA data points: {len(df_nvda)}")
print(f"Date range: {df_nvda['date'].min().date()} to {df_nvda['date'].max().date()}")
print(f"\nFirst few rows:")
df_nvda[['date', 'adj_close', 'adj_volume', 'momentum_6m', 'is_eligible']].head(10)

NVDA data points: 1253
Date range: 2020-12-09 to 2025-12-04

First few rows:


,date,adj_close,adj_volume,momentum_6m,is_eligible
8653176,2020-12-09,12.93075,382954440.0,NaN,False
8653177,2020-12-10,12.97225,205640840.0,NaN,False
8653178,2020-12-11,13.01325,208385320.0,NaN,False
8653179,2020-12-14,13.30875,257076960.0,NaN,False
8653180,2020-12-15,13.36050,190897480.0,NaN,False
8653181,2020-12-16,13.24250,222322080.0,NaN,False
8653182,2020-12-17,13.34125,227522400.0,NaN,False
8653183,2020-12-18,13.27200,327968320.0,NaN,False
8653184,2020-12-21,13.33225,298379120.0,NaN,False
8653185,2020-12-22,13.27825,184506200.0,NaN,False


In [4]:
# Create the chart
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03,
    subplot_titles=('NVDA Split-Adjusted Price', 'Volume'),
    row_heights=[0.7, 0.3]
)

# Candlestick chart (using adj_close for all OHLC since we have adjusted data)
fig.add_trace(
    go.Candlestick(
        x=df_nvda['date'],
        open=df_nvda['adj_open'],
        high=df_nvda['adj_high'],
        low=df_nvda['adj_low'],
        close=df_nvda['adj_close'],
        name='NVDA'
    ),
    row=1, col=1
)

# Volume bars
fig.add_trace(
    go.Bar(
        x=df_nvda['date'],
        y=df_nvda['adj_volume'],
        name='Volume',
        marker_color='rgba(0, 150, 255, 0.5)'
    ),
    row=2, col=1
)

# Update layout
fig.update_layout(
    title='NVDA - Split-Adjusted Price & Volume',
    yaxis_title='Price ($)',
    yaxis2_title='Volume',
    xaxis_rangeslider_visible=False,
    height=800,
    showlegend=False,
    hovermode='x unified'
)

fig.show()

# Export to HTML
fig.write_html('nvda_chart_from_dataset.html')
print("Chart exported to nvda_chart_from_dataset.html")

Chart exported to nvda_chart_from_dataset.html


In [ ]:
# Show NVDA stats
print("\n" + "="*60)
print("NVDA Statistics")
print("="*60)
print(f"Current price: ${df_nvda['adj_close'].iloc[-1]:.2f}")
print(f"All-time high: ${df_nvda['adj_close'].max():.2f}")
print(f"All-time low: ${df_nvda['adj_close'].min():.2f}")
print(f"Latest 6-month momentum: {df_nvda['momentum_6m'].iloc[-1]:.2%}" if pd.notna(df_nvda['momentum_6m'].iloc[-1]) else "Latest 6-month momentum: N/A")
print(f"\nEligibility:")
print(f"  Currently eligible: {df_nvda['is_eligible'].iloc[-1]}")
print(f"  Times eligible: {df_nvda['is_eligible'].sum()}")
print(f"  Times in top 20: {df_nvda['in_top_n'].sum()}")

# Show recent momentum rankings
recent_rebalances = df_nvda[df_nvda['is_rebalance_date'] & df_nvda['is_eligible']].tail(10)
if len(recent_rebalances) > 0:
    print(f"\nRecent rebalancing dates (rank out of ~4,566 eligible stocks):")
    for _, row in recent_rebalances.iterrows():
        rank = int(row['momentum_rank']) if pd.notna(row['momentum_rank']) else 'N/A'
        momentum = f"{row['momentum_6m']:.1%}" if pd.notna(row['momentum_6m']) else 'N/A'
        top20 = "✓" if row['in_top_n'] else ""
        print(f"  {row['date'].date()}: Rank #{rank:>4}, Momentum: {momentum:>7} {top20}")


NVDA Statistics
Current price: $183.38
All-time high: $207.04
All-time low: $11.23
Latest 6-month momentum: 31.00%

Eligibility:
  Currently eligible: True
  Times eligible: 1127
  Times in top 20: 0

Recent rebalancing dates (rank out of ~4,566 eligible stocks):
  2025-07-21: Rank # 536, Momentum:   28.3% 
  2025-08-04: Rank # 226, Momentum:   49.9% 
  2025-08-18: Rank # 484, Momentum:   31.1% 
  2025-09-02: Rank # 444, Momentum:   49.7% 
  2025-09-16: Rank # 588, Momentum:   46.3% 
  2025-09-30: Rank # 497, Momentum:   72.2% 
  2025-10-14: Rank # 722, Momentum:   62.6% 
  2025-10-28: Rank # 449, Momentum:   84.4% 
  2025-11-11: Rank # 615, Momentum:   48.7% 
  2025-11-25: Rank # 944, Momentum:   31.9% 


: 